In [20]:
import pandas as pd
from pathlib import Path
import librosa
import os
import numpy as np

import warnings
import logging
import traceback
from tqdm import tqdm

In [22]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
COUGHS_DIR = RAW_DIR / "coughs"
COUGHVID_DIR = COUGHS_DIR/ "COUGHVID_V3"
TB_DIR = COUGHS_DIR / "TBscreen_Dataset"
WEST_CHINA_DIR = COUGHS_DIR / "West_China_Uni"
BRONCHITIS_DIR = WEST_CHINA_DIR / "bronchitis"
PNEUMONIA_DIR = WEST_CHINA_DIR / "pneumonia"
ARTIFACTS_DIR = ROOT / "artifacts"
PROCESSED_DIR = DATA_DIR / "processed"

In [3]:
def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, low_memory=False)  # mixed-type columns, avoid dtype warnings


# COUGHVID_v3
coughvid_df = load_csv(COUGHVID_DIR / "tabular_form" / "coughvid_v3.csv")
coughvid_extr_features_df = load_csv(COUGHVID_DIR / "tabular_form" / "extracted_features_coughvid_v3.csv")
coughvid_filtered_exp_lbls = load_csv(COUGHVID_DIR / "tabular_form" / "filtered_expert_labels_coughvid_v3.csv")

# TB_screen
tb_forced_df = load_csv(TB_DIR / "Forced_coughs" / "Forced_coughs.csv")
tb_passive_df = load_csv(TB_DIR / "Passive_coughs" / "Passive_coughs.csv")
tb_metadata_df = load_csv(TB_DIR / "metadata.csv")

# West China, no metadata file, labels come from directory names
# (bronchitis / pneumonia), built below in section 3

In [4]:
coughvid_df.columns

Index(['datetime', 'cough_detected', 'latitude', 'longitude', 'age', 'gender',
       'respiratory_condition', 'fever_muscle_pain', 'status', 'file_name',
       'audio_name'],
      dtype='object')

In [5]:
coughvid_df.shape

(34434, 11)

In [6]:
coughvid_df['status'].value_counts()

status
healthy        15476
symptomatic     3873
COVID-19        1315
Name: count, dtype: int64

In [7]:
coughvid_df = coughvid_df.dropna(subset=["status"]).reset_index(drop=True)

In [8]:
coughvid_df.shape

(20664, 11)

In [9]:
coughvid_df['status'].value_counts()

status
healthy        15476
symptomatic     3873
COVID-19        1315
Name: count, dtype: int64

In [10]:
coughvid_df = coughvid_df[coughvid_df['cough_detected'] >= 0.4].reset_index(drop=True)

In [11]:
coughvid_df.shape

(16662, 11)

In [12]:
coughvid_df['status'].value_counts()

status
healthy        12498
symptomatic     3255
COVID-19         909
Name: count, dtype: int64

In [13]:
coughvid_df.columns

Index(['datetime', 'cough_detected', 'latitude', 'longitude', 'age', 'gender',
       'respiratory_condition', 'fever_muscle_pain', 'status', 'file_name',
       'audio_name'],
      dtype='object')

In [14]:
final_coughvid_df = coughvid_df.drop(columns=['datetime', 'age', 'gender', 'respiratory_condition', 'cough_detected', 'latitude', 'longitude', 'fever_muscle_pain', 'file_name'])

In [15]:
final_coughvid_df.columns

Index(['status', 'audio_name'], dtype='object')

In [16]:
covid = final_coughvid_df[final_coughvid_df['status'] == 'COVID-19']
healthy = final_coughvid_df[final_coughvid_df['status'] == 'healthy'].sample(n=1000, random_state=42)

final_coughvid_df = pd.concat([covid, healthy]).reset_index(drop=True)

In [17]:
final_coughvid_df.shape

(1909, 2)

In [18]:
final_coughvid_df.head()

,status,audio_name
0,COVID-19,72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm
1,COVID-19,6a2ffeef-99c5-4765-8dac-92aec8459d79.webm
2,COVID-19,4fa3ac8d-f252-40d6-bccd-24aa5e35556c.webm
3,COVID-19,21d95211-da92-44a9-83c7-30f8d4c6d670.webm
4,COVID-19,dbb52561-93ca-444e-9ff9-e61fd6c6996f.webm


In [35]:
"""
Audio feature extraction pipeline for cough/respiratory classification.

Requires ffmpeg installed and on PATH for .webm/.ogg files:
    Windows: winget install ffmpeg  (then restart terminal/kernel)
    Mac:     brew install ffmpeg
    Linux:   sudo apt install ffmpeg
"""

warnings.filterwarnings("ignore")

logging.basicConfig(
    filename="feature_extraction_errors.log",
    level=logging.ERROR,
    format="%(asctime)s | %(message)s",
)

SAMPLE_RATE = 22050  # default, can still be overridden per call


def build_file_path(audio_folder: Path, name: str, extension: str | None) -> Path:
    if extension and not str(name).endswith(extension):
        name = f"{name}{extension}"
    return audio_folder / name


def extract_features(clip_path: Path, sr: int = SAMPLE_RATE) -> dict | None:
    try:
        y, sr = librosa.load(clip_path, sr=sr)

        if y is None or len(y) == 0:
            logging.error(f"Empty audio after load: {clip_path}")
            return None

        features = {}

        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i, (mean, std) in enumerate(zip(mfccs.mean(axis=1), mfccs.std(axis=1))):
            features[f"mfcc_{i}_mean"] = mean
            features[f"mfcc_{i}_std"] = std

        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features["chroma_mean"] = chroma.mean()
        features["chroma_std"] = chroma.std()

        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        features["spectral_centroid_mean"] = centroid.mean()
        features["spectral_centroid_std"] = centroid.std()

        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features["spectral_rolloff_mean"] = rolloff.mean()
        features["spectral_rolloff_std"] = rolloff.std()

        bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features["bandwidth_mean"] = bandwidth.mean()
        features["bandwidth_std"] = bandwidth.std()

        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features["contrast_mean"] = contrast.mean()
        features["contrast_std"] = contrast.std()

        zcr = librosa.feature.zero_crossing_rate(y)
        features["zcr_mean"] = zcr.mean()
        features["zcr_std"] = zcr.std()

        rms = librosa.feature.rms(y=y)
        features["rms_mean"] = rms.mean()
        features["rms_std"] = rms.std()

        features["duration"] = librosa.get_duration(y=y, sr=sr)

        return features

    except Exception:
        logging.error(f"Failed on {clip_path}\n{traceback.format_exc()}")
        return None


def run_pipeline(
    df: pd.DataFrame,
    audio_dir: Path,
    id_column: str = "audio_name",
    file_extension: str | None = ".webm",
    sr: int = SAMPLE_RATE,
    cache_path: str | Path | None = None,
    output_path: str | Path | None = None,
) -> pd.DataFrame:
    """
    Extract audio features for every row in `df`, matching files in `audio_dir`
    by `id_column`.

    Set `cache_path` to reuse a previously computed CSV instead of recomputing.
    Set `output_path` to save the merged result, if None nothing is written to disk.
    """
    if cache_path and os.path.exists(cache_path):
        print(f"Loading cached features from {cache_path} ...")
        return pd.read_csv(cache_path)

    print("Extracting features, this will take a while on a large dataset...")

    audio_records = []
    missing_files = 0
    failed_files = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
        audio_path = build_file_path(Path(audio_dir), row[id_column], file_extension)

        if not audio_path.exists():
            missing_files += 1
            logging.error(f"Missing file: {audio_path}")
            continue

        features = extract_features(audio_path, sr=sr)

        if features is None:
            failed_files += 1
            continue

        features[id_column] = row[id_column]
        audio_records.append(features)

    print(f"Done. {len(audio_records)} succeeded, {missing_files} missing, {failed_files} failed to decode.")
    print("See feature_extraction_errors.log for details on failures.")

    audio_df = pd.DataFrame(audio_records)
    final_df = df.merge(audio_df, on=id_column, how="inner")

    if output_path:
        final_df.to_csv(output_path, index=False)
        print(f"Saved to {output_path}")

    return final_df

In [ ]:
print(COUGHVID_AUDIO_DIR)
print(final_coughvid_df['audio_name'].iloc[0])

test_path = build_file_path(COUGHVID_AUDIO_DIR, final_coughvid_df['audio_name'].iloc[0], FILE_EXTENSION)
print(test_path)
print(test_path.exists())

print(os.listdir(COUGHVID_AUDIO_DIR)[:5])
final_df = run_pipeline(final_coughvid_df, COUGHVID_AUDIO_DIR)
print(f"final_df shape: {final_coughvid_df.shape}")

c:\Users\emirl\respiratory-disease-screening\data\raw\coughs\COUGHVID_V3\coughvid_20211012
72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm
c:\Users\emirl\respiratory-disease-screening\data\raw\coughs\COUGHVID_V3\coughvid_20211012\72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm
True
['00014dcc-0f06-4c27-8c7b-737b18a2cf4c.json', '00014dcc-0f06-4c27-8c7b-737b18a2cf4c.webm', '00039425-7f3a-42aa-ac13-834aaa2b6b92.json', '00039425-7f3a-42aa-ac13-834aaa2b6b92.webm', '0007c6f1-5441-40e6-9aaf-a761d8f2da3b.json']
Extracting features, this will take a while on a large dataset...


Extracting features: 100%|██████████| 1909/1909 [10:00<00:00,  3.18it/s]


Done. 1677 succeeded, 232 missing, 0 failed to decode.
See feature_extraction_errors.log for details on failures.
Saved to c:\Users\emirl\respiratory-disease-screening\data\processed\final_features.csv
final_df shape: (1909, 2)


In [23]:
final_df.head()

,status,audio_name,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,...,spectral_rolloff_std,bandwidth_mean,bandwidth_std,contrast_mean,contrast_std,zcr_mean,zcr_std,rms_mean,rms_std,duration
0,COVID-19,72ab770e-a11d-4f98-8e94-7027f3f7a0ab.webm,-470.847931,190.612717,42.349316,62.147629,-20.277021,39.468700,15.817849,27.149414,...,2524.126719,1074.424121,1029.427529,16.049184,12.626969,0.087139,0.114379,0.030565,0.053363,9.72
1,COVID-19,6a2ffeef-99c5-4765-8dac-92aec8459d79.webm,-354.059265,170.827591,63.228638,67.722466,-43.616390,58.100597,2.691243,26.449554,...,1710.580791,1565.791825,507.983420,23.105411,14.382338,0.171478,0.095028,0.070727,0.105960,9.96
2,COVID-19,4fa3ac8d-f252-40d6-bccd-24aa5e35556c.webm,-488.905792,152.985077,8.413086,18.446627,-8.186396,19.812853,1.097989,10.272716,...,1544.407638,2953.664652,356.064460,19.292783,11.028386,0.346085,0.211492,0.015557,0.045500,9.60
3,COVID-19,21d95211-da92-44a9-83c7-30f8d4c6d670.webm,-532.129578,149.447968,44.484325,65.836685,-8.647761,29.667706,5.361758,18.435301,...,2472.470332,1658.632607,1010.359113,16.638784,11.042805,0.138599,0.153303,0.012546,0.029861,9.90
4,COVID-19,dbb52561-93ca-444e-9ff9-e61fd6c6996f.webm,-386.770660,206.173264,28.192722,38.605892,-1.467129,30.957123,22.833906,30.141464,...,729.172432,2125.658382,446.617215,23.104090,17.871928,0.405967,0.122298,0.065221,0.081394,9.84


In [25]:
tb_forced_df.head()

,Unnamed: 0,path,subject,device,cohort,sub,Label,age,gndr_a,lab_sptm_xprt_rslt_a,...,comor,cough_cause_other,transfer_from_A,cough_cause,cough_trt,Primary findings,Cavities - Y/N; if Y number of lobes,Number_of_quadrants_with_consolidation,Abnormal_quadrants_number_denominator_4,Permission_sound
0,0,PID_175A_0_codec,PID_175A,codec,A,175.0,TB,36.0,0.0,5.0,...,6.0,NaN,NaN,NaN,NaN,"RUL limited consolidatin, RLL/LUL infiltrate",NO,1,3.0,Yes
1,1,PID_175A_0_pixel,PID_175A,pixel,A,175.0,TB,36.0,0.0,5.0,...,6.0,NaN,NaN,NaN,NaN,"RUL limited consolidatin, RLL/LUL infiltrate",NO,1,3.0,Yes
2,2,PID_175A0_yeti,PID_175A,yeti,A,175.0,TB,36.0,0.0,5.0,...,6.0,NaN,NaN,NaN,NaN,"RUL limited consolidatin, RLL/LUL infiltrate",NO,1,3.0,Yes
3,3,PID_175A_1_codec,PID_175A,codec,A,175.0,TB,36.0,0.0,5.0,...,6.0,NaN,NaN,NaN,NaN,"RUL limited consolidatin, RLL/LUL infiltrate",NO,1,3.0,Yes
4,4,PID_175A_1_pixel,PID_175A,pixel,A,175.0,TB,36.0,0.0,5.0,...,6.0,NaN,NaN,NaN,NaN,"RUL limited consolidatin, RLL/LUL infiltrate",NO,1,3.0,Yes


In [37]:
tb_forced_final = tb_forced_df[['path', 'Label']].copy()

In [38]:
tb_forced_final['Label'].value_counts()

Label
TB     983
NTB    234
Name: count, dtype: int64

In [30]:
final_df['status'].value_counts()

status
healthy     842
COVID-19    835
Name: count, dtype: int64

In [40]:
tb_forced_final.head()

,path,Label
0,PID_175A_0_codec,TB
1,PID_175A_0_pixel,TB
2,PID_175A0_yeti,TB
3,PID_175A_1_codec,TB
4,PID_175A_1_pixel,TB


In [42]:
tb_forced_processed = run_pipeline(df=tb_forced_final, audio_dir=TB_DIR / "Forced_coughs" / "Audio_files", id_column="path", file_extension=".wav", output_path=PROCESSED_DIR / "tb_forced_features.csv")

Extracting features, this will take a while on a large dataset...


Extracting features: 100%|██████████| 1225/1225 [00:44<00:00, 27.45it/s] 

Done. 1018 succeeded, 207 missing, 0 failed to decode.
See feature_extraction_errors.log for details on failures.
Saved to c:\Users\emirl\respiratory-disease-screening\data\processed\tb_forced_features.csv


In [44]:
tb_forced_processed.head()

,path,Label,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,...,spectral_rolloff_std,bandwidth_mean,bandwidth_std,contrast_mean,contrast_std,zcr_mean,zcr_std,rms_mean,rms_std,duration
0,PID_175A_0_codec,TB,-371.212372,280.962738,41.073421,36.096283,-21.477480,22.654772,9.306717,10.659833,...,2912.945588,1683.708260,1179.401209,15.684482,11.398350,0.131658,0.099808,0.041151,0.060205,1.0
1,PID_175A_0_pixel,TB,-387.276337,291.777527,17.530342,20.455729,-17.585833,18.001141,-0.129791,7.677703,...,3765.978481,2002.280356,1384.844078,15.851766,9.761393,0.181041,0.149369,0.037558,0.054846,1.0
2,PID_175A0_yeti,TB,-371.232941,280.953186,41.072414,36.128250,-21.522274,22.602449,9.236334,10.614820,...,2932.921736,1679.447927,1177.129788,15.664279,11.429439,0.130171,0.098754,0.041145,0.060209,1.0
3,PID_175A_1_codec,TB,-466.858521,234.496506,33.357311,41.405006,-9.030167,16.316439,10.824057,13.365821,...,2918.748689,1203.803582,1289.315013,13.915206,9.206522,0.067327,0.079763,0.024017,0.042656,1.0
4,PID_175A_1_pixel,TB,-480.131927,242.244583,19.834051,25.065645,-7.661618,12.939384,5.213512,6.869092,...,3688.571885,1405.852085,1483.475733,15.388040,7.856168,0.089966,0.106837,0.019195,0.033191,1.0


In [53]:
tb_forced_processed['Label'].value_counts()

Label
TB     840
NTB    170
Name: count, dtype: int64

In [46]:
def build_labeled_dataset(
    label_dirs: dict[str, Path],
    output_path: str | Path,
    extensions: tuple[str, ...] = (".wav", ".webm", ".ogg", ".mp3"),
    sr: int = SAMPLE_RATE,
) -> pd.DataFrame:
    """
    Walk each folder in `label_dirs`, extract audio features from every file,
    and save one CSV with a 'label' column and a 'filename' column.

    Example:
        build_labeled_dataset(
            label_dirs={
                "bronchitis": Path("data/bronchitis"),
                "pneumonia": Path("data/pneumonia"),
            },
            output_path="respiratory_features.csv",
        )
    """
    records = []
    missing_or_failed = 0

    for label, folder in label_dirs.items():
        folder = Path(folder)
        files = [f for f in folder.iterdir() if f.suffix.lower() in extensions]

        for audio_path in tqdm(files, desc=f"Extracting {label}"):
            features = extract_features(audio_path, sr=sr)

            if features is None:
                missing_or_failed += 1
                continue

            features["filename"] = audio_path.name
            features["label"] = label
            records.append(features)

    print(f"Done. {len(records)} succeeded, {missing_or_failed} failed to decode.")
    print("See feature_extraction_errors.log for details on failures.")

    final_df = pd.DataFrame(records)
    final_df.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")

    return final_df

In [48]:
west_china_df = build_labeled_dataset(
    label_dirs={
        "bronchitis": Path(BRONCHITIS_DIR),
        "pneumonia": Path(PNEUMONIA_DIR),
    },
    output_path=PROCESSED_DIR / "west_china_features.csv",
)

west_china_df["label"].value_counts()

Extracting pneumonia: 100%|██████████| 82/82 [00:09<00:00,  8.80it/s]

Done. 173 succeeded, 0 failed to decode.
See feature_extraction_errors.log for details on failures.
Saved to c:\Users\emirl\respiratory-disease-screening\data\processed\west_china_features.csv


label
bronchitis    91
pneumonia     82
Name: count, dtype: int64

In [49]:
west_china_df.head()

,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,...,bandwidth_std,contrast_mean,contrast_std,zcr_mean,zcr_std,rms_mean,rms_std,duration,filename,label
0,-189.831131,60.947975,99.795967,13.650887,-44.067696,12.791242,-3.955976,10.974785,-28.117050,9.253667,...,164.326794,20.776892,11.030130,0.118478,0.026084,0.049610,0.046284,1.300000,B1.mp3,bronchitis
1,-269.487823,66.559502,142.031769,25.957262,-23.352831,15.757566,-3.997889,8.074896,-22.315807,11.972027,...,333.506575,20.306535,11.575434,0.081648,0.027816,0.027329,0.027214,2.800000,B10.mp3,bronchitis
2,-260.357697,86.528999,134.211731,31.187517,-21.061752,14.705850,-1.381762,7.629925,-23.298862,13.641817,...,442.231614,20.371478,11.810267,0.080117,0.019158,0.040138,0.046855,3.758776,B11.mp3,bronchitis
3,-180.028854,67.598946,88.113373,47.915703,-38.363308,17.893097,-19.293489,22.854931,-33.959949,12.520788,...,565.176302,19.823959,11.306564,0.134675,0.058461,0.058683,0.057549,1.500000,B12.mp3,bronchitis
4,-215.545792,64.538368,104.724174,38.477150,-33.852596,16.474688,-9.020253,17.819399,-27.037512,12.103587,...,475.437945,20.014559,11.909164,0.121122,0.046975,0.039900,0.044627,3.657143,B13.mp3,bronchitis


In [50]:
west_china_df.columns

Index(['mfcc_0_mean', 'mfcc_0_std', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean',
       'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std',
       'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean',
       'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std',
       'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std',
       'mfcc_12_mean', 'mfcc_12_std', 'chroma_mean', 'chroma_std',
       'spectral_centroid_mean', 'spectral_centroid_std',
       'spectral_rolloff_mean', 'spectral_rolloff_std', 'bandwidth_mean',
       'bandwidth_std', 'contrast_mean', 'contrast_std', 'zcr_mean', 'zcr_std',
       'rms_mean', 'rms_std', 'duration', 'filename', 'label'],
      dtype='object')

In [51]:
final_df.columns

Index(['status', 'audio_name', 'mfcc_0_mean', 'mfcc_0_std', 'mfcc_1_mean',
       'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std',
       'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean',
       'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std',
       'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std',
       'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std',
       'chroma_mean', 'chroma_std', 'spectral_centroid_mean',
       'spectral_centroid_std', 'spectral_rolloff_mean',
       'spectral_rolloff_std', 'bandwidth_mean', 'bandwidth_std',
       'contrast_mean', 'contrast_std', 'zcr_mean', 'zcr_std', 'rms_mean',
       'rms_std', 'duration'],
      dtype='object')

In [54]:
tb_forced_processed.columns

Index(['path', 'Label', 'mfcc_0_mean', 'mfcc_0_std', 'mfcc_1_mean',
       'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std',
       'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean',
       'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std',
       'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std',
       'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std',
       'chroma_mean', 'chroma_std', 'spectral_centroid_mean',
       'spectral_centroid_std', 'spectral_rolloff_mean',
       'spectral_rolloff_std', 'bandwidth_mean', 'bandwidth_std',
       'contrast_mean', 'contrast_std', 'zcr_mean', 'zcr_std', 'rms_mean',
       'rms_std', 'duration'],
      dtype='object')